# 텍스트 cell - probe 기반 FULL sweep (분류 3 + 회귀 3)

finetune(무거움->dev만 가능) 대신 **이미지 cell과 동일한 frozen DistilBERT 임베딩 + probe**(ADR-019 방식)로
성능을 재서, 전 6데이터셋 x 전 오염단계를 실제로 완주한다. -> "텍스트 full sweep"을 둔갑 아닌 실제 실행으로 달성.

- 분류(ag_news/imdb/20news): F1(macro) / 회귀(yelp/amazon/sst5): R2(0 clip)
- 오염은 train만, 평가는 clean test / train·test 항상 disjoint(누수 방지)

**덮어쓰기 방지**: 결과는 신규 파일 `results/text_probe_metrics.csv` 에만 기록. 기존 CSV는 읽기 전용. per-config 체크포인트+resume.

실행: 위에서부터 셀 순서대로. 30분 권장값 TRAIN_N=800 / TEST_N=250 (셀 안 숫자만 조절).


In [1]:
# ============================================================
# 0. Colab 셋업 + BASE 탐색 (Drive 마운트)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%pip -q install datasets transformers

import os, sys, glob, time
import numpy as np
import pandas as pd

def _find_base():
    env = os.environ.get('DSC_BASE')
    if env and os.path.isfile(f'{env}/dsc_framework/__init__.py'):
        return env
    root = '/content/drive/MyDrive'
    for c in [f'{root}/capstone/dsc', f'{root}/dsc', f'{root}/capstone-dsc']:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pat in [f'{root}/*/dsc_framework/__init__.py', f'{root}/*/*/dsc_framework/__init__.py']:
        for hit in glob.glob(pat):
            return os.path.dirname(os.path.dirname(hit))
    raise RuntimeError('dsc_framework/ 못 찾음 - Drive 마운트/동기화 확인')

BASE = _find_base()
RESULTS = f'{BASE}/results'
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('BASE =', BASE)


Mounted at /content/drive
BASE = /content/drive/MyDrive/capstone/dsc


In [2]:
# ============================================================
# 1. 덮어쓰기 방지 가드 + 출력 경로 (★ 새 파일에만 기록)
# ============================================================
OUT = f'{RESULTS}/text_probe_metrics.csv'        # 유일한 쓰기 대상 (신규)
DSC_SWEEP = f'{RESULTS}/text_dsc_sweep.csv'       # 읽기 전용 입력
PROTECTED = {'text_train_metrics.csv', 'text_train_metrics_dev.csv', 'text_dsc_sweep.csv',
             'text_baseline_dsc.csv', 'dsc_scores.csv', 'model_performance.csv', 'merged_results.csv'}
assert os.path.basename(OUT) not in PROTECTED, f'OUT={OUT} 가 보호 파일과 충돌 - 중단'

# ============================================================
# 2. 설정 (30분 하드캡 권장값; 숫자만 바꿔 조절)
# ============================================================
TRAIN_N = int(os.environ.get('TEXT_TRAIN_N', 800))
TEST_N  = int(os.environ.get('TEXT_TEST_N', 250))
BATCH   = int(os.environ.get('TEXT_BATCH', 64))
SEED    = 42
LEVELS  = [0.1, 0.25, 0.5, 0.75, 0.9]   # text_dsc_sweep.csv 와 동일한 비-baseline 단계

# ============================================================
# 3. imports - 기존 모듈 재사용 (DSC 임베딩 / probe / polluter)
# ============================================================
from dsc_framework.text_cell import _extract_features
from dsc_framework.perf_probe import evaluate_probes
from dsc_framework.text_polluters import (
    CompletenessTextPolluter, NoiseInjectionTextPolluter, WordShufflePolluter,
    ClassBalanceTextPolluter, LabelSwapTextPolluter,
    TargetDistributionSkewTextPolluter, TargetNoiseTextPolluter,
)

# polluter 이름은 text_dsc_sweep.csv 의 polluter 컬럼과 정확히 일치 (merge 키)
POLLUTERS_CLS = {
    'completeness_text':    CompletenessTextPolluter,
    'noise_injection_text': NoiseInjectionTextPolluter,
    'word_shuffle':         WordShufflePolluter,
    'class_balance':        ClassBalanceTextPolluter,
    'label_swap':           LabelSwapTextPolluter,
}
POLLUTERS_REG = {
    'completeness_text':        CompletenessTextPolluter,
    'noise_injection_text':     NoiseInjectionTextPolluter,
    'word_shuffle':             WordShufflePolluter,
    'target_distribution_skew': TargetDistributionSkewTextPolluter,
    'target_noise':             TargetNoiseTextPolluter,
}

DATASET_SPECS = [
    ('ag_news',   'fancyzhx/ag_news',               'classification'),
    ('imdb',      'stanfordnlp/imdb',               'classification'),
    ('20news',    'SetFit/20_newsgroups',           'classification'),
    ('yelp_full', 'Yelp/yelp_review_full',          'regression'),
    ('amazon_en', 'SetFit/amazon_reviews_multi_en', 'regression'),
    ('sst5',      'SetFit/sst5',                    'regression'),
]

# ============================================================
# 4. helpers - train/test 분할은 항상 disjoint (누수 방지)
# ============================================================
def _to_lists(ds_split, task):
    texts = list(ds_split['text'])
    labels = list(ds_split['label'])
    labels = [float(y) for y in labels] if task == 'regression' else [int(y) for y in labels]
    return texts, labels

def _subsample(ds_split, n, task, seed=42):
    if n is not None and len(ds_split) > n:
        ds_split = ds_split.shuffle(seed=seed).select(range(n))
    return _to_lists(ds_split, task)

def _load_train_test(ds, task):
    """test split 있으면 사용(서로 다른 split -> 자동 disjoint).
    없으면 train 을 셔플해 앞쪽 TEST_N / 그 다음 TRAIN_N 으로 잘라 disjoint 보장."""
    if 'test' in ds:
        tr_t, tr_y = _subsample(ds['train'], TRAIN_N, task, seed=SEED)
        te_t, te_y = _subsample(ds['test'],  TEST_N,  task, seed=SEED)
        return tr_t, tr_y, te_t, te_y
    full = ds['train'].shuffle(seed=SEED)
    n_te = min(TEST_N, max(1, len(full) // 5))
    te_t, te_y = _to_lists(full.select(range(n_te)), task)
    upper = min(n_te + TRAIN_N, len(full))
    tr_t, tr_y = _to_lists(full.select(range(n_te, upper)), task)
    return tr_t, tr_y, te_t, te_y

# ============================================================
# 5. resume - 기존 OUT(신규 파일)에서 끝난 config skip
# ============================================================
done = set()
if os.path.isfile(OUT):
    _prev = pd.read_csv(OUT)
    done = set(zip(_prev['dataset'], _prev['polluter'], _prev['level'].astype(float)))
    print(f'[resume] 기존 {OUT} 에서 {len(done)} config 완료 - skip')

def _append_rows(rows):
    df = pd.DataFrame(rows)
    header = not os.path.isfile(OUT)
    df.to_csv(OUT, mode='w' if header else 'a', header=header, index=False)

# ============================================================
# 6. sweep (config 단위 try/except - 하나 실패해도 전체 진행)
# ============================================================
def main():
    from datasets import load_dataset
    t0 = time.time(); n_done = n_skip = n_err = 0
    for ds_name, hf_id, task in DATASET_SPECS:
        print(f'\n=== {ds_name} ({task}) - load {hf_id} ===', flush=True)
        try:
            ds = load_dataset(hf_id)
            tr_texts, tr_y, te_texts, te_y = _load_train_test(ds, task)
        except Exception as e:
            print(f'    [SKIP dataset] {ds_name} 로드 실패: {e}', flush=True); continue
        print(f'    train={len(tr_texts)} test={len(te_texts)}', flush=True)
        Xte, _ = _extract_features(te_texts, sample_cap=None, batch_size=BATCH)
        yte = np.asarray(te_y, dtype=float if task == 'regression' else int)

        polluters = POLLUTERS_CLS if task == 'classification' else POLLUTERS_REG
        configs = [('none', 0.0)] + [(p, lv) for p in polluters for lv in LEVELS]
        for pname, lv in configs:
            key = (ds_name, pname, float(lv))
            if key in done:
                n_skip += 1; continue
            try:
                if pname == 'none':
                    ptr_texts, ptr_y = list(tr_texts), list(tr_y)
                else:
                    ptr_texts, ptr_y = polluters[pname](level=lv, random_seed=SEED).pollute(
                        list(tr_texts), list(tr_y))
                if len(ptr_texts) < 5:
                    print(f'    [skip] {pname}_{lv}: 오염 후 표본부족({len(ptr_texts)})'); continue
                Xtr, _ = _extract_features(ptr_texts, sample_cap=None, batch_size=BATCH)
                ytr = np.asarray(ptr_y, dtype=float if task == 'regression' else int)
                scores = evaluate_probes(Xtr, ytr, Xte, yte, task)
                rows = [{'dataset': ds_name, 'polluter': pname, 'level': float(lv),
                         'method': 'probe', 'model': k, 'score': v, 'task': task}
                        for k, v in scores.items() if not k.startswith('_') and v is not None]
                if rows:
                    _append_rows(rows); done.add(key); n_done += 1
                    _mean = np.mean([r['score'] for r in rows])
                    print(f'    {pname:24s} lv={lv:<4} probe평균={_mean:+.3f} '
                          f'({n_done} done, {time.time()-t0:.0f}s)', flush=True)
            except Exception as e:
                n_err += 1
                print(f'    [ERR] {ds_name}/{pname}_{lv}: {type(e).__name__}: {e}', flush=True)
    print(f'\n[완료] 신규 {n_done}, skip {n_skip}, 에러 {n_err}. {time.time()-t0:.0f}s -> {OUT}')
    _quick_scoreboard()

def _quick_scoreboard():
    """text_dsc_sweep.csv(읽기 전용)와 merge -> 데이터셋별 r 출력. 파일 안 씀."""
    if not (os.path.isfile(OUT) and os.path.isfile(DSC_SWEEP)):
        print('(scoreboard 생략: 파일 없음)'); return
    try:
        from scipy.stats import pearsonr, spearmanr
    except Exception:
        print('(scipy 없음 - scoreboard 생략)'); return
    perf = pd.read_csv(OUT)
    dsc = pd.read_csv(DSC_SWEEP)
    pm = perf.groupby(['dataset', 'polluter', 'level'])['score'].mean().reset_index(name='probe_mean')
    score_col = 'dsc_score' if 'dsc_score' in dsc.columns else ('score' if 'score' in dsc.columns else None)
    if score_col is None:
        print('(text_dsc_sweep.csv 점수 컬럼 없음 - 생략)'); return
    dsc_s = dsc[['dataset', 'polluter', 'level', score_col]].drop_duplicates(['dataset', 'polluter', 'level'])
    m = pm.merge(dsc_s, on=['dataset', 'polluter', 'level'])
    print('\n=== 데이터셋별 r (DSC <-> probe 평균) ===')
    for ds_name, sub in m.groupby('dataset'):
        if len(sub) < 4:
            print(f'  {ds_name:12s} n={len(sub):2d} (부족)'); continue
        rp, _ = pearsonr(sub[score_col], sub['probe_mean'])
        rs, _ = spearmanr(sub[score_col], sub['probe_mean'])
        print(f'  {ds_name:12s} n={len(sub):2d}  pearson={rp:+.3f}  spearman={rs:+.3f}  [{"PASS" if rp>=0.40 else "below"}]')


In [3]:
# === 실행: 전 6데이터셋 probe sweep (결과는 text_probe_metrics.csv 에만) ===
main()



=== ag_news (classification) - load fancyzhx/ag_news ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...

test.jsonl:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8544 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1101 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2210 [00:00<?, ? examples/s]

    train=800 test=250
    none                     lv=0.0  probe평균=+0.187 (131 done, 3396s)
    completeness_text        lv=0.1  probe평균=+0.152 (132 done, 3432s)
    completeness_text        lv=0.25 probe평균=+0.123 (133 done, 3470s)
    completeness_text        lv=0.5  probe평균=+0.092 (134 done, 3509s)
    completeness_text        lv=0.75 probe평균=+0.058 (135 done, 3550s)
    completeness_text        lv=0.9  probe평균=+0.001 (136 done, 3597s)
    noise_injection_text     lv=0.1  probe평균=+0.116 (137 done, 3638s)
    noise_injection_text     lv=0.25 probe평균=+0.041 (138 done, 3678s)
    noise_injection_text     lv=0.5  probe평균=+0.008 (139 done, 3721s)
    noise_injection_text     lv=0.75 probe평균=+0.000 (140 done, 3764s)
    noise_injection_text     lv=0.9  probe평균=+0.000 (141 done, 3808s)
    word_shuffle             lv=0.1  probe평균=+0.154 (142 done, 3844s)
    word_shuffle             lv=0.25 probe평균=+0.132 (143 done, 3882s)
    word_shuffle             lv=0.5  probe평균=+0.113 (144 done, 3919

In [4]:
# === scoreboard만 다시 보기 ===
_quick_scoreboard()



=== 데이터셋별 r (DSC <-> probe 평균) ===
  20news       n=25  pearson=+0.629  spearman=+0.762  [PASS]
  ag_news      n=25  pearson=+0.698  spearman=+0.869  [PASS]
  amazon_en    n=25  pearson=+0.500  spearman=+0.702  [PASS]
  imdb         n=25  pearson=+0.409  spearman=+0.618  [PASS]
  sst5         n=25  pearson=+0.516  spearman=+0.704  [PASS]
  yelp_full    n=25  pearson=+0.584  spearman=+0.740  [PASS]
